# 🤖 RoboBarista – der FAIRINO FR5 führt die Gießgeste mit sechs Gelenken aus!

**Nächster Meilenstein nach den Kameras:** Das echte virtuelle FR5-Gelenkmodell bewegt ein **starr am Handgelenk befestigtes Milchkännchen**. Die Bewegung wird über inverse Kinematik in sechs reale Gelenkwinkel pro Frame übersetzt. MuJoCo berechnet danach die tatsächliche Kännchenposition durch Vorwärtskinematik – wir **teleportieren das Kännchen nicht mehr**.

**Du bekommst:** einen GIF-Film des FR5 bei der Bewegung, einen Soll-Ist-Plot mit Positions-/Winkelfehlern, einen Bericht über Gelenkgrenzen und erkannte Kontaktpaare sowie Zeitreihe aller sechs Gelenke als JSON.

**Was das noch nicht ist:** Kein echtes Greifen, kein dynamisch mit Motoren gesteuerter Roboter, keine Kraftregelung, keine Geschwindigkeits-/Beschleunigungsfreigabe, kein tatsächliches Milchgießen. Die Kännchenhalterung ist eine **starre visuelle Testbefestigung**; deshalb ignoriert das Kännchen noch physische Kollisionen. Das ist **kinematischer FR5-Gelenk-Dry-Run** mit dem bestehenden synthetischen Bewegungsmuster, nicht kameragemessene Latte-Art-Demonstration.

**So geht's:** Im Menü **Laufzeit → Alle ausführen** wählen. Google Colab benötigt hierfür keine GPU (CPU + OSMesa).


In [ ]:
# 1. GitHub-Projekt laden (keine ZIP nötig)
import os, subprocess, sys, pathlib
repo = pathlib.Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Coffee-Tobi/robocasa.git', str(repo)], check=True)
else:
    # Bei erneutem Ausführen aktuelle Verbesserungen von GitHub holen.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Projekt:', repo)


In [ ]:
# 2. Systemgrafik (CPU/OSMesa) und Python-Pakete installieren
# Ohne GPU benötigt MuJoCo echte OSMesa-Systembibliotheken; pip allein reicht nicht.
import os, subprocess, sys, ctypes.util
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6-dev',
                'libgl1', 'libglx-mesa0', 'libgl1-mesa-dri'], check=True)
osmesa_lib = ctypes.util.find_library('OSMesa')
print('OSMesa-Systembibliothek:', osmesa_lib)
if not osmesa_lib:
    raise RuntimeError('OSMesa fehlt trotz apt-Installation. Bitte die Ausgabe von Schritt 2 teilen.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements.txt'], check=True)

# Eigenständiger Importcheck im richtigen Rendering-Backend (ohne GPU: OSMesa).
backend = 'egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
check_env = os.environ.copy()
check_env['MUJOCO_GL'] = backend
if backend == 'osmesa':
    check_env['PYOPENGL_PLATFORM'] = 'osmesa'
print('MuJoCo-Importtest mit Backend:', backend, flush=True)
subprocess.run([sys.executable, '-c',
                'import mujoco; print("MuJoCo:", mujoco.__version__, "Import OK")'],
               env=check_env, check=True)
import cv2, numpy as np
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)


In [ ]:
# 3. Mathematische Tests – vollständige Fehleranzeige statt kryptischem Exit-Code
import subprocess,sys
test = subprocess.run(
    [sys.executable, '-m', 'pytest', '-vv', '-ra', '--tb=short', 'tests'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, errors='replace', check=False)
print('=== Testergebnisse (stdout) ===', flush=True)
print(test.stdout[-30000:] if test.stdout.strip() else '(leer)', flush=True)
print('=== Zusätzliche Fehlermeldungen (stderr) ===', flush=True)
print(test.stderr[-20000:] if test.stderr.strip() else '(leer)', flush=True)
if test.returncode:
    raise RuntimeError(
        'Mindestens ein Python-Test ist fehlgeschlagen. '
        'Oben steht der genaue Testname unter FAILURES/ERRORS; '
        'bitte diesen Abschnitt teilen. Nicht mit unvalidierten Messungen fortfahren.')
print('Alle Python-Tests bestanden.')


In [ ]:
# 4. Erstes echtes FR5-Gelenk-Replay mit virtuellem Kännchen
import os, sys, subprocess
from pathlib import Path
env=os.environ.copy()
env['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if env['MUJOCO_GL']=='osmesa':
    env['PYOPENGL_PLATFORM']='osmesa'
cmd=[sys.executable,'-u','-m','robobarista.robot_replay',
     '--frames','25','--fps','8','--width','960','--height','720',
     '--output','runs/fr5_replay']
print('Backend:',env['MUJOCO_GL'])
print('Start: 25 virtuelle Zeitpunkte, FAIRINO FR5 sechs Gelenke',flush=True)
process=subprocess.Popen(cmd,env=env,cwd=str(repo),text=True,
                         errors='replace',stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT,bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
exit_code=process.wait()
print('Exit-Code:',exit_code)
if exit_code:
    print('Der Bericht und die Bilder koennen trotz nicht bestandener IK- oder Kontaktpruefung existieren.')
    print('Schau dir in Schritt 5 die Diagnose an und teile uns die Ausgabe mit.')


In [ ]:
# 5. Animation, Fehlergrafik und echter Gelenkstatus
from IPython.display import display, Image
from pathlib import Path
import json, pandas as pd
folder=Path(repo)/'runs/fr5_replay'
gif=folder/'fr5_replay.gif'
if gif.exists():
    display(Image(filename=str(gif),format='gif'))
else:
    print('Kein GIF erzeugt. Bitte Fehlermeldung aus Schritt 4 pruefen.')
graph=folder/'ik_error.png'
if graph.exists(): display(Image(filename=str(graph)))
summary=folder/'summary.json'
if summary.exists():
    s=json.loads(summary.read_text())
    print('Status:',s['status'])
    print('Frames mit erfolgreicher 6-Gelenk-IK:',s['successful_ik_frames'],'von',s['frames'])
    print('Groesster Positionsfehler (mm):',s['max_pitcher_position_error_mm'])
    print('Groesster Winkelabweichung (Grad):',s['max_pitcher_orientation_error_deg'])
    print('Groesster Ausgiesserspitzen-Fehler (mm):',s['max_spout_error_mm'])
    print('Gelenkgrenzen verletzt:',s['joint_limit_failures'])
    print('Robot-Umgebung-Kontaktpaare gesamt:',s['robot_environment_contacts'])
    print('Selbst-Kontaktpaare gesamt:',s['robot_self_contacts'])
    print('\nEinschraenkung: Das Kännchen ist starr und rein visuell befestigt, kein dynamischer Greiftest.')
    display(pd.read_csv(folder/'ik_metrics.csv').head(25))
else:
    print('Kein summary.json gefunden; bitte Ausgabe aus Schritt 4 senden.')


In [ ]:
# 6. Optional: Video/IK-Daten als ZIP herunterladen
import zipfile
from IPython.display import FileLink,display
if (folder/'summary.json').exists():
    target=folder/'robobarista_fr5_replay.zip'
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for file in folder.rglob('*'):
            if file.is_file() and file!=target:
                z.write(file,file.relative_to(folder))
    display(FileLink(str(target)))


## Danach: Kameradaten statt synthetischer Wunschbewegung

Sobald dieses Notebook durchläuft und der FR5 sichtbar das Kännchen bewegt, können wir **`trajectory.csv` aus [COLAB_MOTION](https://colab.research.google.com/github/Coffee-Tobi/robocasa/blob/main/COLAB_MOTION.ipynb)** als Bewegungsquelle wählen. Anschließend ergänzen wir Gelenkgeschwindigkeits-/Beschleunigungsgrenzen und eine **echte Kollisionsprüfung des Kännchens**, bevor wir über Motorregelung oder physischen Griff sprechen.

[Projekt auf GitHub](https://github.com/Coffee-Tobi/robocasa) · [Alle Projektmeilensteine](https://github.com/Coffee-Tobi/robocasa/blob/main/planning/ROBOBARISTA_DREI_STUFEN_MVP.md)